# ====================== Başlıq ======================

# Azərbaycan Nitq Tanıma Sistemi (ASR) - Part A
**Model:** LocalDoc/azerbaijani-whisper-small

**Dataset:** Mozilla Common Voice (az) - test split

## 

# ====================== Quraşdırma ======================

In [ ]:
import os
import glob
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from evaluate import load
import librosa
import pandas as pd
from tqdm import tqdm
import re

##
## 

# ====================== Dataset ======================

In [ ]:
for root, dirs, files in os.walk(r"C:\Users\USER\Documents\Task_RISC"):
    level = root.replace(r"C:\Users\USER\Documents\Task_RISC", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}[{os.path.basename(root)}]")
    subindent = "  " * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

In [ ]:
data_path = "../az"

tsv_files = glob.glob(os.path.join(data_path, "*.tsv"))

dataframes = {}

for file_path in tsv_files:
    file_name = os.path.basename(file_path).replace(".tsv", "")
    df = pd.read_csv(file_path, sep='\t')
    dataframes[f"{file_name}_df"] = df
    
    print(f"--- {file_name.upper()} DATASET INFO ---")
    print(df.info())
    print("\n")

In [ ]:
data_path = "../az"

train_df = pd.read_csv(os.path.join(data_path, "train.tsv"), sep='\t')
dev_df = pd.read_csv(os.path.join(data_path, "dev.tsv"), sep='\t')
test_df = pd.read_csv(os.path.join(data_path, "test.tsv"), sep='\t')

for df in [train_df, dev_df, test_df]:
    df['full_path'] = df['path'].apply(lambda x: os.path.join(data_path, "clips", x))

print(f"Train nümunə sayı: {len(train_df)}")
print(f"Dev nümunə sayı: {len(dev_df)}")
print(f"Test nümunə sayı: {len(test_df)}")

In [ ]:
clips_count = len(os.listdir("../az/clips"))
print(f"Clips qovluğundakı toplam səs faylı sayı: {clips_count}")

# ====================== MODEL ======================


In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "LocalDoc/azerbaijani-whisper-small"   

processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)

wer_metric = load("wer")
cer_metric = load("cer")


# ================= NORMALIZATION ======================


In [ ]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\səöüğıçş]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ====================== TRANSCRIBE ======================


In [ ]:
def transcribe_audio(audio_path):
    speech, sr = librosa.load(audio_path, sr=16000)
    
    inputs = processor(speech, sampling_rate=16000, return_tensors="pt").to(device)
    
    forced_ids = processor.get_decoder_prompt_ids(language="az", task="transcribe")
    
    predicted_ids = model.generate(
        inputs.input_features,
        forced_decoder_ids=forced_ids,     
        max_length=448,        
        temperature=0.0,
        do_sample=False,
        num_beams=1
    )
    
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcription

# ====================== EVALUATION ======================


In [ ]:
test_df = pd.read_csv(os.path.join(data_path, "test.tsv"), sep='\t')
test_df['path'] = test_df['path'].apply(lambda x: os.path.join(data_path, "clips", x))

test_sample = test_df.sample(126, random_state=42)   

predictions = []
references = []

for _, row in tqdm(test_sample.iterrows(), total=len(test_sample)):
    pred = transcribe_audio(row['path'])
    predictions.append(pred)
    references.append(row['sentence'])

clean_pred = [normalize_text(p) for p in predictions]
clean_ref = [normalize_text(r) for r in references]

wer = wer_metric.compute(predictions=clean_pred, references=clean_ref)
cer = cer_metric.compute(predictions=clean_pred, references=clean_ref)

print(f"WER: {wer*100:.2f}%")
print(f"CER: {cer*100:.2f}%")



In [ ]:
results = []
for p, r in zip(clean_pred, clean_ref):
    wer_i = wer_metric.compute(predictions=[p], references=[r])
    results.append({"reference": r, "prediction": p, "wer": wer_i})

results_sorted = sorted(results, key=lambda x: x["wer"])

print("\n=== ƏN YAXŞI 5 NÜMUNƏ ===")
for item in results_sorted[:5]:
    print(f"WER: {item['wer']:.3f} | Ref: {item['reference']} | Pred: {item['prediction']}\n")

print("=== ƏN PİS 5 NÜMUNƏ ===")
for item in results_sorted[-5:]:
    print(f"WER: {item['wer']:.3f} | Ref: {item['reference']} | Pred: {item['prediction']}\n")
